# Risk Parameters for BTC

In [21]:
import os
from typing import Tuple
import numpy as np
import pandas as pd
import plotly.express as px
pd.options.plotting.backend = "plotly"

import sys
sys.path.append(os.getcwd().split("scripts")[0])
sys.path.append(os.path.join(os.getcwd().split("scripts")[0], "params"))

import params
from params import funding, caps
import pystable

from scipy.stats import levy_stable

## Loading and Analysing the Original Data

In [22]:
filename = "btc"

path_to_file = os.path.join(os.getcwd().split("scripts")[0], f"data/{filename}")

periodicity = 60. # 1 minute in seconds
cap = 10  # cap on pay off, set by governance

short_twap = 10 * periodicity
long_twap = 60 * periodicity

periodicity, short_twap, long_twap

(60.0, 600.0, 3600.0)

In [23]:
df = pd.read_csv(path_to_file+".csv", parse_dates=["timestamp"]).set_index("timestamp")
df

,close
timestamp,
2024-01-01 00:01:00+00:00,42298.61
2024-01-01 00:02:00+00:00,42320.00
2024-01-01 00:03:00+00:00,42325.50
2024-01-01 00:04:00+00:00,42367.99
2024-01-01 00:05:00+00:00,42397.23
...,...
2024-05-08 13:06:00+00:00,62097.67
2024-05-08 13:07:00+00:00,62077.49
2024-05-08 13:08:00+00:00,62115.51


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 185110 entries, 2024-01-01 00:01:00+00:00 to 2024-05-08 13:10:00+00:00
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   close   185110 non-null  float64
dtypes: float64(1)
memory usage: 2.8 MB


Looking at the indexes

In [27]:
df.index.is_unique, df.index.is_monotonic_increasing

(True, True)

In [28]:
diff = np.diff(df.index.to_numpy())
print(f"Is index equally spaced? {'YES' if np.all(diff == diff[0]) else 'NO'} ")

Is index equally spaced? YES 


In [ ]:
def detect_consecutive_entries(_df, _column, epsilon:float=1e-12):
    condition = (_df[_column].diff().abs() < epsilon).any()
    print(f"Does column '{_column}' contains consecutive entries? {'YES' if condition else 'NO'} ")

In [30]:
detect_consecutive_entries(df, _column="close", epsilon=1e-15)

Does column 'close' contains consecutive entries? YES 


In [106]:
def count_consecutive_entries(_df, _column, epsilon:float=1e-12):
    return (_df[_column].diff().abs() < epsilon).sum()

In [107]:
count_consecutive_entries(df, _column="close", epsilon=1e-15)

3372

## TWAP

Computing the 1h TWAP, sampled every 10 min.

In [86]:
df_twap = df.rolling(60).mean().dropna().resample('10min', label="right").last()
df_twap

,close,log_return
timestamp,,
2024-01-01 01:10:00+00:00,42445.579667,0.000011
2024-01-01 01:20:00+00:00,42443.685833,0.000005
2024-01-01 01:30:00+00:00,42450.079333,0.000033
2024-01-01 01:40:00+00:00,42485.129333,0.000140
2024-01-01 01:50:00+00:00,42528.335833,0.000094
...,...,...
2024-05-08 12:40:00+00:00,62331.672000,-0.000036
2024-05-08 12:50:00+00:00,62302.007667,-0.000084
2024-05-08 13:00:00+00:00,62279.337333,-0.000012


In [87]:
df_twap.plot()

In [31]:
detect_consecutive_entries(df_twap, _column="close", epsilon=1e-12)

Does column 'close' contains consecutive entries? NO 


## Computing Log Returns

Definition:

$$
r_t = \ln(1+R_t)= \ln \dfrac{P_t}{P_{t-1}} = p_t − p_{t−1}
$$

In [32]:
def compute_log_return(_df:pd.DataFrame, column:str) -> pd.DataFrame:
    tmp_df = _df.copy()
    if not column in tmp_df.columns:
        raise Exception(f"Column {column} not found")
    tmp_df["log_return"] = np.nan
    tmp_df.loc[tmp_df.index[1]:,"log_return"] = np.log(
        tmp_df[column].iloc[1:].to_numpy() / tmp_df[column].iloc[:-1].to_numpy()
    )
    tmp_df.dropna(inplace=True)
    return tmp_df

TWAP log return

In [33]:
df_twap = compute_log_return(df_twap, column="close")
df_twap

,close,log_return
timestamp,,
2024-01-01 01:20:00+00:00,42443.685833,-0.000045
2024-01-01 01:30:00+00:00,42450.079333,0.000151
2024-01-01 01:40:00+00:00,42485.129333,0.000825
2024-01-01 01:50:00+00:00,42528.335833,0.001016
2024-01-01 02:00:00+00:00,42562.376667,0.000800
...,...,...
2024-05-08 12:40:00+00:00,62331.672000,0.000106
2024-05-08 12:50:00+00:00,62302.007667,-0.000476
2024-05-08 13:00:00+00:00,62279.337333,-0.000364


Log return of the original data

In [34]:
df = compute_log_return(df, column="close")
df

,close,log_return
timestamp,,
2024-01-01 00:02:00+00:00,42320.00,0.000506
2024-01-01 00:03:00+00:00,42325.50,0.000130
2024-01-01 00:04:00+00:00,42367.99,0.001003
2024-01-01 00:05:00+00:00,42397.23,0.000690
2024-01-01 00:06:00+00:00,42409.20,0.000282
...,...,...
2024-05-08 13:06:00+00:00,62097.67,-0.000543
2024-05-08 13:07:00+00:00,62077.49,-0.000325
2024-05-08 13:08:00+00:00,62115.51,0.000612


## Fit Distribution

In [88]:
def fit_distribution(
    _df:pd.DataFrame, column:str="close", period:float=periodicity, verbose:bool=False
) -> Tuple[pystable.STABLE_DIST, pystable.STABLE_DIST]:

    if not column in _df.columns:
        raise Exception(f"Column {column} not found")

    dst = funding.gaussian()
    pystable.fit(dst, _df[column].to_numpy(), _df.index.size)

    if verbose:
        print(f'''
            alpha: {dst.contents.alpha}, beta: {dst.contents.beta},
            mu: {dst.contents.mu_1}, sigma: {dst.contents.sigma}
            '''
        )

    scaled_dst = funding.rescale(dst, 1./period)
    scaled_dst_2 = caps.rescale(dst, 1./period)

    if verbose:
        print(f'''
            rescaled params (1/t = {1./period}):
            alpha: {scaled_dst.contents.alpha}, beta: {scaled_dst.contents.beta},
            mu: {scaled_dst.contents.mu_1}, sigma: {scaled_dst.contents.sigma}
            '''
        )
        print(f'''
            rescaled params (1/t = {1./period}):
            alpha: {scaled_dst_2.contents.alpha}, beta: {scaled_dst_2.contents.beta},
            mu: {scaled_dst_2.contents.mu_1}, sigma: {scaled_dst_2.contents.sigma}
            '''
        )

    return dst, scaled_dst

Fit original data

In [89]:
dst, scaled_dst = fit_distribution(
    df, column="log_return", period=periodicity, verbose=True
)


            alpha: 1.4398185687536873, beta: 0.03143063846574884,
            mu: 1.035529684203383e-05, sigma: 0.0003567229771301606
            

            rescaled params (1/t = 0.016666666666666666):
            alpha: 1.4398185687536873, beta: 0.03143063846574884,
            mu: 1.7258828070056382e-07, sigma: 2.07657789339447e-05
            

            rescaled params (1/t = 0.016666666666666666):
            alpha: 1.4398185687536873, beta: 0.03143063846574884,
            mu: 1.7258828070056382e-07, sigma: 2.6748413659098222e-05
            


Fit TWAP

In [90]:
dst_twap, scaled_dst_twap = fit_distribution(
    df_twap, column="log_return", period=10*periodicity, verbose=True
)


            alpha: 1.4291860218861046, beta: 0.04157197539035555,
            mu: 3.3831069467459034e-06, sigma: 4.433224933708221e-05
            

            rescaled params (1/t = 0.0016666666666666668):
            alpha: 1.4291860218861046, beta: 0.04157197539035555,
            mu: 5.63851157790984e-09, sigma: 5.044872263827972e-07
            

            rescaled params (1/t = 0.0016666666666666668):
            alpha: 1.4291860218861046, beta: 0.04157197539035555,
            mu: 5.63851157790984e-09, sigma: 6.476876578321519e-07
            


## Compare Distribution and Data

In [91]:
def compare_dist_and_data(
    _df:pd.DataFrame,
    _column:str,
    _any_dst:pystable.STABLE_DIST,
    _bins:int=1000
) -> pd.DataFrame:

    min_1pct = pystable.q(_any_dst, [0.01], 1)[0]
    max_99pct = pystable.q(_any_dst, [0.99], 1)[0]

    pdf, bin_edges = np.histogram(
        _df[_column].to_numpy(), bins=np.linspace(min_1pct, max_99pct, _bins+1), density=True
    )

    x = np.linspace(min_1pct, max_99pct, _bins)

    df_any_dst = pd.DataFrame(
        pystable.pdf(_any_dst, x, len(x)), 
        columns=["dist_pdf"], index=x
    )

    df_any_dst["dist_cdf"] = pystable.cdf(_any_dst, x, len(x))

    df_any_dst["data_pdf"] = pdf
    df_any_dst["data_cdf"] = np.cumsum(pdf * np.diff(bin_edges))

    return df_any_dst

In [92]:
df_dst_dist = compare_dist_and_data(df, "log_return", dst, _bins=5000)

df_scaled_dist = compare_dist_and_data(df, "log_return", scaled_dst, _bins=5000)

In [112]:
df.index[0].strftime("%Y-%m-%d")

'2024-01-01'

In [125]:
df_dst_dist[["dist_cdf", "data_cdf"]].plot(
    title=f"CDF of BTCUSDT data from {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}"
)

In [126]:
df_scaled_dist[["dist_cdf", "data_cdf"]].plot(
    title=f"Rescaled CDF (scale:{1/periodicity:.4f}) of BTCUSDT data from {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}"
)

In [119]:
df_dst_dist[["dist_pdf", "data_pdf"]].plot(
    title=f"PDF of BTCUSDT data from {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}"
)

In [120]:
df_scaled_dist[["dist_pdf", "data_pdf"]].plot(
    title=f"Rescaled PDF (scale:{1/periodicity:.4f}) of BTCUSDT data from {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}"
)

In [121]:
df_twap_dst_dist = compare_dist_and_data(df_twap, "log_return", dst_twap, _bins=1000)

df_twap_scaled_dist = compare_dist_and_data(df_twap, "log_return", scaled_dst_twap, _bins=1000)

In [123]:
df_twap_dst_dist[["dist_cdf", "data_cdf"]].plot(
    title=f"CDF of hourly TWAP BTCUSDT, 10min sample, from {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}"
)

In [124]:
df_twap_dst_dist[["dist_pdf", "data_pdf"]].plot(
    title=f"PDF of hourly TWAP BTCUSDT, 10min sample, from {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}"
)

## Funding Rate